# Projet : Stage
### Objectif

Pour les attributs spatiaux et temporels, on voudrait faire la même chose. Pour cette partie, il y a 3 étapes à faire : 

1. Identifier les attributs spatiaux / temporels. 
2. Trouver le niveau d'hiérarchie de chaque attributs selon les hiérarchies spatiales / temporelles
3. Identifier le niveau d'hiérarchie le plus fin parmis tous les attributs spatiaux / temporels en tant que granularité minimum de dataset; identifier l'attribut au niveau d'hiérarchie le plus haut parmis tous les attributs en tant que scope de dataset et donner la liste de ses valeurs distinctes. 

L'output final qu'on demande est un dossier json de métadonnée de tous les datasets.

## 1. Hiérarchisation des données

### 1.1. Arrondissement/Cantons/Communes/Departements/Regions

In [4]:
import json

In [5]:
def fillChamp(dicChamps, dic_hierarchisee, rang) :

    for i in range(len(dicChamps['features'])) :
        champ = dicChamps['features'][i]['properties']['nom']
        if champ not in dic_hierarchisee[rang] :
            dic_hierarchisee[rang].append(champ.lower())
        
    return 

def fillDictionnaire() :
    fichiers = ['arrondissements', 'cantons', 'communes', 'departements', 'regions']
    dic_hierarchisee = {}

    for fichier in fichiers :
        dic_hierarchisee[fichier] = []
        mon_json = open(f"Education/levels/france-geojson/{fichier}-avec-outre-mer.geojson")
        data = json.load(mon_json)
        mon_json.close()

        fillChamp(data, dic_hierarchisee, fichier)

    return dic_hierarchisee

In [6]:
dic_hierarchisee = fillDictionnaire()

### 1.2 Quartiers

In [7]:
import csv

In [8]:

fichier = open("liste-correspondance-qp2024-qp2015.csv", "r", encoding="utf-8")
reader = csv.reader(fichier, delimiter=";")
listeQuartiers = list(reader)[1:]

In [9]:
dic_hierarchisee['quartiers'] = []
for i in range(len(listeQuartiers)) :
    quartier = listeQuartiers[i][1]
    if quartier not in dic_hierarchisee['quartiers'] and quartier != "" :
        dic_hierarchisee['quartiers'].append(quartier.lower())

dic_hierarchisee['QP'] = []
for i in range(len(listeQuartiers)) :
    quartier = listeQuartiers[i][3]
    if quartier not in dic_hierarchisee['QP'] and quartier != "" :
        dic_hierarchisee['QP'].append(quartier.lower())

del i, listeQuartiers, quartier, fichier, reader

In [10]:
for key in dic_hierarchisee:
    dic_hierarchisee[key].sort()

del key

## 2. Identification des attributs spatiaux dans un fichier csv/xlsx

### 2.1 Algorithme de recherche

Mon objectif est de parcourir un tableau en vérifiant à chaque cellule si elle appartient à une donnée de mon dictionnaire jusqu'à trouver la plus petite et la plus grande granularité

In [11]:
# Tableau rangé par granularité des champs
champs = ['QP', 'quartiers', 'arrondissements', 'cantons', 'communes', 'departements', 'regions']
champs.reverse()
dic_hierarchisee = {champ: dic_hierarchisee[champ] for champ in champs if champ in dic_hierarchisee}

### 2.2 Fichiers csv & xlsx

In [16]:
import os
import pandas as pd

def getFiles(origine):
    fichiers = []
    for dossier in os.walk(origine):
        for fichier in dossier[2]:
            if fichier.endswith('.csv') or fichier.endswith('.xlsx'):
                fichiers.append(os.path.join(dossier[0], fichier))
    return fichiers

documents = getFiles("Opendata/Général/Département")

In [17]:
hierarchie_champs = {champ: i for i, champ in enumerate(dic_hierarchisee.keys())}

total_fichier = 0

for fichier in documents:
    if fichier.endswith('.csv'):
        total_fichier += 1

print(f"Nombre de fichiers CSV : {total_fichier}")

Nombre de fichiers CSV : 1


In [18]:
dico_spatialscope = {}
compteur = 0
for fichier in documents :
    score_colonne = {}
    nom_fic = fichier.split("/")[-1]
    liste_attributs_spatiaux = {}
    low_gran_and_scope = {}

    if fichier.endswith('csv') :
        compteur += 1
        print(f"Traitement du fichier {nom_fic}... | {compteur}/{total_fichier} fichiers traités")
        fic = open(fichier, "r")
        tab = csv.reader(fic, delimiter=";")
        headers = next(tab)
        for k in range (len(headers)) :
            score_colonne[k] = 0

        if len(headers) <= 1 :
            tab = csv.reader(fic, delimiter=",")
            headers = headers[0].split(",")
        
        tab = list(tab)
        fic.close()

        for i in range(100) :
            for j in range (len(tab[i])) :
                if isinstance(tab[i][j], str) :
                    tab[i][j] = tab[i][j].lower()
                for attribut, donnees in dic_hierarchisee.items() :
                    if tab[i][j] in donnees and tab[i][j] != "":
                        score_colonne[j] += 1
                        if headers[j] not in liste_attributs_spatiaux and attribut not in liste_attributs_spatiaux.values() and score_colonne[j] > 90 :
                            liste_attributs_spatiaux[headers[j]] = attribut
        
        if len(liste_attributs_spatiaux) == 0 :
            le_plus_bas = None
            le_plus_haut = None

        else :
    
            le_plus_bas = ['regions', None]
            le_plus_haut = ['QP', None]
            for attribut, champ in liste_attributs_spatiaux.items() :
                # print(f"Test en cours : {attribut} dans {champ} inférieur à {le_plus_bas}")
                if hierarchie_champs[champ] > hierarchie_champs[le_plus_bas[0]] :
                    le_plus_bas = [champ, attribut]
                # print(f"Test en cours : {attribut} dans {champ} supérieur à {le_plus_haut}")
                if hierarchie_champs[champ] < hierarchie_champs[le_plus_haut[0]] :
                    le_plus_haut = [champ, attribut]
            
        low_gran_and_scope[nom_fic] = {'Low granularity': le_plus_bas, 'Scope': le_plus_haut}

        # Preparation du dictionnaire pour le fichier de metadonnees
        if low_gran_and_scope[nom_fic]['Low granularity'] != None and low_gran_and_scope[nom_fic]['Scope'] != None :
            colonne = headers.index(low_gran_and_scope[nom_fic]['Scope'][1])
            donnees = []
            for ligne in tab :
                if ligne[colonne].lower() not in donnees :
                    donnees.append(ligne[colonne].lower())
            dico_spatialscope[nom_fic] = {"spatialScopeLevel": low_gran_and_scope[nom_fic]['Scope'][0], "spatialScope": donnees}
# del attribut, champ, champs, data, df, documents, donnees, fic, fichier, headers, i, j, le_plus_bas, le_plus_haut, sheet_name, tab, donnees

Traitement du fichier temperature-quotidienne-departementale.csv... | 1/1 fichiers traités


In [ ]:
# Enregistrement du dictionnaire dans un fichier JSON
with open("dico_spatialscope.json", "w", encoding="utf-8") as f:
    json.dump(dico_spatialscope, f, ensure_ascii=False, indent=4)

## 3. Création des classes utiles a la structure de métadonnée par fichier

In [ ]:
import uml_class

# Classes UML du fichier de metadonnees
fic_spatialscope = uml_class.DS_Spatial_Scope(None, None).from_dict(dico_spatialscope[nom_fic])
# fic_temporalscope = uml_class.DS_Temporal_Scope()
# fic_theme = uml_class.Theme()
# fic_datacontent = uml_class.Data_Content()
# fic_parameter = uml_class.Parameter()
# fic_spatialparam = uml_class.Spatial_Parameter()
# fic_temporalparam = uml_class.Temporal_Parameter()
# fic_complementary = uml_class.Complementary_Information()
# fic_indicator = uml_class.Existing_Indicator()
# fic_dataset = uml_class.Dataset()
